# Load Themes from Rebrickable CSV Download
Downloads `themes.csv.gz` from the Rebrickable bulk download page and loads it into a Spark DataFrame.

In [0]:
import io
import gzip
import requests
import pandas as pd
from pyspark.sql import SparkSession
from pyspark.sql.types import StructType, StructField, IntegerType, StringType

DOWNLOAD_URL = "https://cdn.rebrickable.com/media/downloads/themes.csv.gz"

In [0]:
response = requests.get(DOWNLOAD_URL, timeout=60)
response.raise_for_status()

with gzip.open(io.BytesIO(response.content)) as f:
    df_pd = pd.read_csv(f)

print(f"Downloaded {len(df_pd)} rows")
print(df_pd.dtypes)
df_pd.head()

In [0]:
schema = StructType([
    StructField("id",        IntegerType(), nullable=False),
    StructField("name",      StringType(),  nullable=True),
    StructField("parent_id", IntegerType(), nullable=True),  # NULL for top-level themes
])

rows = [
    (
        int(row["id"]),
        row["name"] if pd.notna(row["name"]) else None,
        int(row["parent_id"]) if pd.notna(row["parent_id"]) else None,
    )
    for _, row in df_pd.iterrows()
]

spark = SparkSession.builder.getOrCreate()
df_themes = spark.createDataFrame(rows, schema=schema)

df_themes.printSchema()
df_themes.display(10, truncate=False)

In [0]:
from datetime import datetime
from pyspark.sql.functions import current_timestamp, lit

# Generate timestamp components for partitioned path and filename
now = datetime.utcnow()
year  = now.strftime("%Y")
month = now.strftime("%m")
day   = now.strftime("%d")
ts    = now.strftime("%Y%m%d_%H%M%S")

# Add audit columns
df_themes_out = (
    df_themes
    .withColumn("_load_datetime", current_timestamp())
    .withColumn("_record_source", lit("CSV_DOWNLOAD"))
)

# Update the base path below to match your Unity Catalog external volume mount point
EXTERNAL_VOLUME_BASE = "/Volumes/lego_brickbase/bronze/raw_data_volume/themes"
PARTITION_PATH       = f"{EXTERNAL_VOLUME_BASE}/{year}/{month}/{day}/{ts}"
TEMP_PATH            = f"{PARTITION_PATH}/_tmp"
FILE_NAME            = f"raw_themes_{ts}.parquet"
FINAL_PATH           = f"{PARTITION_PATH}/{FILE_NAME}"

# Write as a single Parquet file to a temp staging directory
df_themes_out.coalesce(1) \
    .write \
    .mode("overwrite") \
    .parquet(TEMP_PATH)

# Locate the single part file and rename it to the desired filename
part_files = [f.path for f in dbutils.fs.ls(TEMP_PATH) if f.name.startswith("part-")]
dbutils.fs.mv(part_files[0], FINAL_PATH)
dbutils.fs.rm(TEMP_PATH, recurse=True)

print(f"Themes written to: {FINAL_PATH}")